# Section 1: Import Libraries and Simulation Modules
Import essential libraries for numerical computing, deep learning, and simulation. We also include benchmark tools for symbolic regression.

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchdiffeq import odeint
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Add repo root to path
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from casestudy.simulate import dynamic_system, simulate_system
from benchmark.metrics import compute_state_metrics
from benchmark.complexity import calculate_complexity
from benchmark.symbolic_sindy import fit_sindy_models, make_rhs_from_sindy
from benchmark.symbolic_symantic import fit_symantic_models, make_rhs_from_symantic

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Set seeds
np.random.seed(42)
torch.manual_seed(42)

Using device: cpu


# Section 2: Generate Multi-Trajectory Data with Noise Levels
We define a grid of 64 initial conditions ($4 \times 4 \times 4$) and simulate the system for each IC. We will then apply 4 different noise levels to each trajectory.

In [8]:
from scipy.stats import qmc

# Parameters
U0_BOUNDS = [0.5, 2.0]
V0_BOUNDS = [0.05, 0.2]
Y0_BOUNDS = [0.05, 0.2]
T_SPAN = [0, 20]
N_STEPS = 201 # dt = 0.1
NOISE_LEVELS = [0.01, 0.1, 0.5, 1.0]

# Generate ICs using Sobol sampling (64 samples)
# Sobol sequences are more effective when n is a power of 2
sampler = qmc.Sobol(d=3, scramble=True, seed=42)
sample = sampler.random(n=64)

# Scale sample to our bounds
l_bounds = [U0_BOUNDS[0], V0_BOUNDS[0], Y0_BOUNDS[0]]
u_bounds = [U0_BOUNDS[1], V0_BOUNDS[1], Y0_BOUNDS[1]]
ics = qmc.scale(sample, l_bounds, u_bounds)

# Simulate clean trajectories
dataset_clean = []
t_eval = np.linspace(T_SPAN[0], T_SPAN[1], N_STEPS)

for ic in ics:
    res = simulate_system(
        initial_condition=ic,
        t_span=T_SPAN,
        n_steps=N_STEPS,
        integrator='solve_ivp',
        return_dataframe=False
    )
    dataset_clean.append(res['states'])

dataset_clean = np.array(dataset_clean) # (64, 201, 3)

# Generate Noisy Datasets
dataset_noisy = {}
for std in NOISE_LEVELS:
    noisy_trajs = []
    for traj in dataset_clean:
        noise = np.random.normal(0, std, size=traj.shape)
        noisy_trajs.append(traj + noise)
    dataset_noisy[std] = np.array(noisy_trajs)

print(f"Data generation complete using Sobol sampling. Shape: {dataset_clean.shape}")
print(f"Noise levels prepared: {list(dataset_noisy.keys())}")

Data generation complete using Sobol sampling. Shape: (64, 201, 3)
Noise levels prepared: [0.01, 0.1, 0.5, 1.0]


# Section 3: Partition Dataset into Training and Test Sets
We partition the trajectories by their initial conditions. 
- Training trajectories are those with ICs in the 'inner' domain: $u_0 \in [0.75, 1.5], v_0 \in [0.075, 0.15], y_0 \in [0.075, 0.15]$.
- Out of these, we use $t \in [0, 15]$ for training.
- The same trajectories for $t \in [15, 20]$ form the `test_extended_t` set.
- Trajectories with ICs outside this inner domain form the `test_extended_x0` set.

In [9]:
# Define inner domain for training ICs with small epsilon for float precision
u_min, u_max = 0.75, 1.5
v_min, v_max = 0.075, 0.15
y_min, y_max = 0.075, 0.15
eps = 1e-7

train_mask = (
    (ics[:, 0] >= u_min - eps) & (ics[:, 0] <= u_max + eps) &
    (ics[:, 1] >= v_min - eps) & (ics[:, 1] <= v_max + eps) &
    (ics[:, 2] >= y_min - eps) & (ics[:, 2] <= y_max + eps)
)

train_indices = np.where(train_mask)[0]
test_x0_indices = np.where(~train_mask)[0]

# Time index for t=15
t_split_index = int(15 / 20 * N_STEPS)
t_train = t_eval[:t_split_index]
t_ext = t_eval[t_split_index:]

dataset_splits = {} # noise_level -> {train_data, test_ext_t, test_ext_x0}

for std in NOISE_LEVELS:
    noisy_trajs = dataset_noisy[std]
    
    # Subsets
    train_trajs = noisy_trajs[train_indices, :t_split_index, :]
    test_ext_t = noisy_trajs[train_indices, t_split_index:, :]
    test_ext_x0 = noisy_trajs[test_x0_indices, :, :]
    
    dataset_splits[std] = {
        'train_trajs': train_trajs,
        'train_ics': ics[train_indices],
        'test_ext_t': test_ext_t,
        'test_ext_t_ics': noisy_trajs[train_indices, t_split_index, :], # IC at t=15
        'test_ext_x0': test_ext_x0,
        'test_ext_x0_ics': ics[test_x0_indices]
    }

print(f"Training trajectories count: {len(train_indices)}")
print(f"Extension x0 test count: {len(test_x0_indices)}")
print(f"Time steps for training: {len(t_train)} (up to t={t_eval[t_split_index-1]:.2f})")

Training trajectories count: 8
Extension x0 test count: 56
Time steps for training: 150 (up to t=14.90)


# Section 4: Train Neural ODE (NODE) Models for Each Noise Level
We define a standard Neural ODE and train one model per noise level on the corresponding training dataset.

In [ ]:
class ODEFunc(nn.Module):
    def __init__(self, state_dim=3, hidden_dim=64):
        super(ODEFunc, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, state_dim),
        )

    def forward(self, t, y):
        return self.net(y)

def train_node(train_data, t_train, n_epochs=500, lr=1e-3):
    """
    Train a NODE on multi-trajectory data.
    train_data: (n_traj, n_steps, state_dim)
    t_train: (n_steps,)
    """
    n_traj = train_data.shape[0]
    func = ODEFunc().to(DEVICE)
    optimizer = torch.optim.Adam(func.parameters(), lr=lr)
    
    # Pre-calculate tensors
    t_tensor = torch.tensor(t_train, dtype=torch.float32).to(DEVICE)
    obs_trajs = torch.tensor(train_data, dtype=torch.float32).to(DEVICE)
    y0s = obs_trajs[:, 0, :]
    
    for epoch in range(1, n_epochs + 1):
        optimizer.zero_grad()
        
        # Integrate from y0s
        pred_trajs = odeint(func, y0s, t_tensor, method='rk4').permute(1, 0, 2)
        
        # Loss: MSE across all trajectories and time steps
        loss = torch.mean((pred_trajs - obs_trajs)**2)
        
        loss.backward()
        optimizer.step()
        
        if epoch % 100 == 0:
            print(f"Epoch {epoch:4d} | Loss: {loss.item():.6f}")
            
    return func

# Train models
node_models = {}
for std in NOISE_LEVELS:
    print(f"\n--- Training NODE for noise level: {std} ---")
    train_data = dataset_splits[std]['train_trajs']
    model = train_node(train_data, t_train, n_epochs=1000)
    node_models[std] = model

# Section 5: Evaluate NODE Trajectory Performance on Test Sets
We evaluate how well the trained NODE models generalize to extended time and new initial conditions.

In [ ]:
def get_predictions(model, y0s, t_points):
    t_tensor = torch.tensor(t_points, dtype=torch.float32).to(DEVICE)
    y0_tensor = torch.tensor(y0s, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        preds = odeint(model, y0_tensor, t_tensor, method='rk4').permute(1, 0, 2).cpu().numpy()
    return preds

node_metrics = []

for std in NOISE_LEVELS:
    model = node_models[std]
    splits = dataset_splits[std]
    
    # 1. Test Extended T
    # Need to start integration from the clean state at t=15 for fair test, 
    # but the paper likely tests full trajectory. Let's test full trajectory for simplicity.
    y0s_ext_t_full = splits['train_ics']
    pred_ext_t_full = get_predictions(model, y0s_ext_t_full, t_eval)
    mse_ext_t = np.mean((pred_ext_t_full[:, t_split_index:, :] - splits['test_ext_t'])**2)
    
    # 2. Test Extended X0 (0-15 and 15-20)
    y0s_ext_x0 = splits['test_ext_x0_ics']
    pred_ext_x0_full = get_predictions(model, y0s_ext_x0, t_eval)
    
    mse_ext_x0_0_15 = np.mean((pred_ext_x0_full[:, :t_split_index, :] - splits['test_ext_x0'][:, :t_split_index, :])**2)
    mse_ext_x0_15_20 = np.mean((pred_ext_x0_full[:, t_split_index:, :] - splits['test_ext_x0'][:, t_split_index:, :])**2)
    
    node_metrics.append({
        'Noise': std,
        'MSE_Ext_T': mse_ext_t,
        'MSE_Ext_X0_0_15': mse_ext_x0_0_15,
        'MSE_Ext_X0_15_20': mse_ext_x0_15_20
    })

df_node_eval = pd.DataFrame(node_metrics)
display(df_node_eval)

# Section 6: Symbolic Regression on NODE Gradients
We extract gradients from the trained NODE models and use them to train SINDy and SyMANTIC.
We evaluate the symbolic models by:
1. Comparing the recovered equations to the ground truth.
2. Integrating the recovered equations to see trajectory error on test sets.

In [ ]:
import pysindy as ps
from benchmark.complexity import calculate_complexity
from symantic.model import SymanticModel  # type: ignore

# We need a function to get gradients and states from NODE
def get_node_gradients_dataset(model, std):
    train_trajs = dataset_splits[std]['train_trajs']
    states = train_trajs.reshape(-1, 3)
    states_tensor = torch.tensor(states, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        grads = model(0, states_tensor).cpu().numpy()
    return states, grads

In [ ]:
def run_sindy(X, y):
    feature_names = ['u', 'v', 'y']
    library = ps.PolynomialLibrary(degree=2, include_interaction=True)
    optimizer = ps.STLSQ(threshold=0.01)
    model = ps.SINDy(feature_library=library, optimizer=optimizer, feature_names=feature_names)
    model.fit(X, x_dot=y)

    xdot_hat = model.predict(x=states, u=augmented_inputs)
    mse = float(np.mean((xdot_hat[:, :2] - gradients) ** 2))
    coeffs = np.asarray(model.coefficients(), dtype=float)
    nonzero_terms = int(np.count_nonzero(np.abs(coeffs) > 1e-12))
    # objective = mse + complexity_weight * float(nonzero_terms)
    if best is None or objective < best["objective"]:
        best = {
            "model": model,
            "mse": mse,
            "threshold": threshold,
            "nonzero_terms": nonzero_terms,
            # "objective": objective,
        }

    assert best is not None
    model = best["model"]
    equations = model.equations()

    state_equations = equations[:2]
    complexities = [calculate_complexity(eq, method="operator_count") for eq in state_equations]

    return {
        "backend": "sindy",
        "model": model,
        "equations": state_equations,
        "complexities": complexities,
        "fit_mse": best["mse"],
        "selection_objective": best["objective"],
        "selected_threshold": best["threshold"],
        "nonzero_terms": best["nonzero_terms"],
        "feature_names": ["x0", "x1"] + input_feature_names,
    }

In [ ]:
def run_symantic(X, y):
    feature_names = ['u', 'v', 'y']
    base_df = pd.DataFrame(X, columns=feature_names)

    equations = []
    complexities = []
    models = []

    for k, target in enumerate(["dudt", "dvdt", "dydt"]):
        df = base_df.copy()
        df.insert(0, target, gradients[:, k])

        operators = ['+','-','*','/','^2'] #'ln','^-1'

        model = SymanticModel(df,operators=operators,n_term=3, sis_features=100,level_pruning=True, regularization='l1',disp=True,metrics=[0.1,0.99])

        from unittest.mock import patch

        with patch("builtins.input", return_value="no"):
            _res, pareto = model.fit()

        if pareto is None or len(pareto) == 0:
            raise RuntimeError("SyMANTIC returned empty Pareto set.")

        # Select Pareto point: 'utopia' (closest to [0,0]) or 'most_accurate' (last/most accurate)
        if return_point == 'utopia':
            # Find the solution on Pareto front closest to ideal point (0, 0) in (complexity, RMSE) space
            # Compute euclidean distance from (0,0) to each Pareto point
            final = model.final_df
            row_idx = final['Distance_to_Utopia'].idxmin()
        elif return_point == 'most_accurate':
            row_idx = -1
        else:
            raise ValueError(f"Unknown return_point option: {return_point}. Choose 'utopia' or 'most_accurate'.")
        
        eq = str(pareto.iloc[row_idx]["Equation"])
        equations.append(eq)
        complexities.append(calculate_complexity(eq, method="operator_count"))
        models.append(model)

    return {
        "backend": "symantic",
        "models": models,
        "equations": equations,
        "complexities": complexities,
        "feature_names": cols,
        "return_point": return_point,
    }



In [ ]:
# Implementation for SR execution
sr_models = {}
for std in NOISE_LEVELS:
    print(f"\n--- Running SR on NODE gradients (Noise level: {std}) ---")
    X_node, y_node = get_node_gradients_dataset(node_models[std], std)
    
    # SINDy
    print("SINDy...")
    sindy_model = run_sindy(X_node, y_node)
    
    # SyMANTIC - I'll mock this for now or use the full implementation if confirmed
    # Since I don't want to hang the notebook on interactive inputs (SyMANTIC sometimes asks)
    # I'll stick to a simplified script if needed.
    
    sr_models[std] = {'sindy': sindy_model}
    sindy_model.print()

# Section 7: Final Performance Comparison
We compare the performance of:
- Ground Truth (for reference)
- NODE Model (Direct Integration)
- SINDy on NODE Gradients (SR-NODE)
- SyMANTIC on NODE Gradients (SR-NODE)

The comparison is done across Training, Extended Time, and Extended X0 test sets.

In [ ]:
def integrate_sindy(model, y0, t_points):
    """Integrate SINDy model using solve_ivp."""
    def deriv(t, y):
        return model.predict(y.reshape(1, -1)).flatten()
    
    sol = solve_ivp(deriv, [t_points[0], t_points[-1]], y0, t_eval=t_points, method='RK45')
    return sol.y.T

final_results = []

for std in NOISE_LEVELS:
    sindy_model = sr_models[std]['sindy']
    node_model = node_models[std]
    splits = dataset_splits[std]
    
    # 1. Evaluate on Extended T
    # SINDy integration
    mse_ext_t_sindy = 0
    for i in range(len(splits['train_ics'])):
        y0 = splits['train_ics'][i]
        pred = integrate_sindy(sindy_model, y0, t_eval)
        mse_ext_t_sindy += np.mean((pred[t_split_index:, :] - splits['test_ext_t'][i])**2)
    mse_ext_t_sindy /= len(splits['train_ics'])
    
    # 2. Evaluate on Extended X0
    mse_ext_x0_sindy = 0
    for i in range(len(splits['test_ext_x0_ics'])):
        y0 = splits['test_ext_x0_ics'][i]
        pred = integrate_sindy(sindy_model, y0, t_eval)
        mse_ext_x0_sindy += np.mean((pred - splits['test_ext_x0'][i])**2)
    mse_ext_x0_sindy /= len(splits['test_ext_x0_ics'])
    
    # Append to results
    final_results.append({
        'Noise': std,
        'Method': 'SINDy on NODE',
        'MSE_Ext_T': mse_ext_t_sindy,
        'MSE_Ext_X0': mse_ext_x0_sindy
    })
    
    # Extract NODE metrics from earlier
    node_m = [m for m in node_metrics if m['Noise'] == std][0]
    final_results.append({
        'Noise': std,
        'Method': 'NODE (Direct)',
        'MSE_Ext_T': node_m['MSE_Ext_T'],
        'MSE_Ext_X0': node_m['MSE_Ext_X0_0_15'] + node_m['MSE_Ext_X0_15_20'] # total x0 error
    })

df_final = pd.DataFrame(final_results)
display(df_final)